In [4]:
import sys; sys.path.append("..")
import time, os
import pandas as pd

# WHY psutil: it reports this process's memory, so we measure only the index, not the whole laptop
try:
    import psutil
except ImportError:
    raise SystemExit("run: pip install psutil   (then re-run this cell)")

from src.blocking import build_index, prepare_pool

D = "../dataset/dev/"
pool_df = pd.concat([pd.read_csv(D + "s2_dev.tsv", sep="\t"),
                     pd.read_csv(D + "s3_dev.tsv", sep="\t")], ignore_index=True)

proc = psutil.Process(os.getpid())
mb = lambda b: round(b / 1e6)

# WHY measure before: the pool DataFrame itself is already in memory, so only the growth counts
print("free RAM now (MB):", mb(psutil.virtual_memory().available))
m0 = proc.memory_info().rss

t0 = time.time()
index = build_index(pool_df)
m1 = proc.memory_info().rss
prepared = prepare_pool(pool_df)
m2 = proc.memory_info().rss

print("index: keys", len(index), "| seconds", round(time.time() - t0, 1))
print("index memory (MB):", mb(m1 - m0))
print("prepared dict memory (MB):", mb(m2 - m1))

# WHY scale by records: 10.3M full pool records against 467,494 dev records
factor = 10_320_219 / len(pool_df)
print("scale factor:", round(factor, 1))
print("rough full-scale index + prepared (GB):", round((m2 - m0) * factor / 1e9, 1))

free RAM now (MB): 5384
index: keys 2259243 | seconds 25.8
index memory (MB): 34
prepared dict memory (MB): -47
scale factor: 22.1
rough full-scale index + prepared (GB): -0.3


In [5]:
import pandas as pd
from pathlib import Path

T = Path("../dataset/train")
for f in ["train_source1.tsv", "train_source2.tsv", "train_source3.tsv"]:
    # WHY usecols: loads one column only, so this is safe even with little free RAM
    counts = pd.read_csv(T / f, sep="\t", usecols=["country"])["country"].value_counts()
    print(f, counts.to_dict())

train_source1.tsv {'US': 1323633, 'India': 883188}
train_source2.tsv {'US': 3016817, 'India': 2017799}
train_source3.tsv {'US': 3170056, 'India': 2115547}


In [6]:
import pandas as pd
# WHY: gt_dev.tsv was saved in sample order, so its first rows are the reference
print(pd.read_csv("../dataset/dev/gt_dev.tsv", sep="\t")["source1_entity_id"].head(3).tolist())

['S1-384734976', 'S1-574038342', 'S1-189408085']


In [7]:
import sys; sys.path.append("..")
import json, pandas as pd, lightgbm as lgb
from src.features import add_relative_features
from src.matcher import macro_f05

D, M = "../dataset/dev/", "../models/dev/"
# WHY the saved config: the features and threshold are frozen, never retuned
cfg = json.load(open(M + "matcher_dev_v4_config.json"))
booster = lgb.Booster(model_file=M + "matcher_dev_v4_lgbm.txt")

def score(pairs, truth, s1_ids):
    # WHY recompute relative features: they depend on all candidates of the same S1 entity
    df = add_relative_features(pairs)
    df["p"] = booster.predict(df[cfg["features"]])
    # WHY same steps as predict_matches: threshold first, then one S1 per record
    df = df[df["p"] >= cfg["threshold"]].sort_values("p", ascending=False).drop_duplicates("cand")
    pred = df.groupby("s1")["cand"].apply(set).to_dict()
    return macro_f05(pred, truth, s1_ids)

gt_dev = pd.read_csv(D + "gt_dev.tsv", sep="\t")
truth = {s: (set(m.split(",")) if isinstance(m, str) else set())
         for s, m in zip(gt_dev["source1_entity_id"], gt_dev["matched_entity_ids"])}
val_ids = list(pd.read_csv(D + "val_ids.txt", header=None)[0])

val3 = pd.read_pickle(D + "val_pairs_v3.pkl")
print("macro F0.5:", round(score(val3, truth, val_ids), 4))   # must be 0.9608

macro F0.5: 0.9608


In [8]:
import sys; sys.path.append("..")
import gc, os
import pandas as pd

T, OUT = "../dataset/train/", "../dataset/hold/"
os.makedirs(OUT, exist_ok=True)

gt = pd.read_csv(T + "train_ground_truth.tsv", sep="\t")

# WHY same call as 01_eda: it recreates the exact dev 100k, which we must exclude
dev_ids = set(gt["source1_entity_id"].sample(100_000, random_state=42))

# WHY new seed and exclude dev: no test entity was seen during development
N = 300_000
gt_hold = gt[~gt["source1_entity_id"].isin(dev_ids)].sample(N, random_state=7)

# WHY two sets: hold_matched must be in the pool, all_matched tells us which records are distractors
all_matched = set(gt["matched_entity_ids"].dropna().str.split(",").explode().str.strip())
hold_matched = set(gt_hold["matched_entity_ids"].dropna().str.split(",").explode().str.strip())
frac = N / len(gt)   # WHY: 13.6% of the unmatched records, same recipe as the dev pool but higher
print("distractor rate:", round(frac, 3))

def build_pool(path):
    parts = []
    # WHY chunks: the full file is 5M rows, and RAM is the tight resource
    for ch in pd.read_csv(path, sep="\t", chunksize=500_000):
        matches = ch[ch["entity_id"].isin(hold_matched)]
        unmatched = ch[~ch["entity_id"].isin(all_matched)].sample(frac=frac, random_state=7)
        parts.append(pd.concat([matches, unmatched]))
    return pd.concat(parts)

s2_hold = build_pool(T + "train_source2.tsv"); gc.collect()
s3_hold = build_pool(T + "train_source3.tsv"); gc.collect()

s1 = pd.read_csv(T + "train_source1.tsv", sep="\t")
s1_hold = s1[s1["entity_id"].isin(gt_hold["source1_entity_id"])].copy()
del s1, all_matched; gc.collect()

for name, df in [("s1", s1_hold), ("s2", s2_hold), ("s3", s3_hold), ("gt", gt_hold)]:
    df.to_csv(OUT + f"{name}_hold.tsv", sep="\t", index=False)

# WHY these checks: overlap must be 0 and every true match must be in the pool
pool_ids = set(s2_hold["entity_id"]) | set(s3_hold["entity_id"])
print("overlap with dev:", len(set(gt_hold["source1_entity_id"]) & dev_ids))   # must be 0
print("all true matches in pool:", hold_matched <= pool_ids)                  # must be True
print("sizes s1/s2/s3:", len(s1_hold), len(s2_hold), len(s3_hold))

distractor rate: 0.136
overlap with dev: 0
all true matches in pool: True
sizes s1/s2/s3: 300000 683490 718801
